### Fase 1: Transformando Canciones en "Canastas"
El algoritmo Apriori no lee DataFrames normales; necesita una lista de transacciones. 
En nuestro caso, cada **Canción** es una transacción, y los **Artistas** que participan en ella son los artículos de la canasta.

In [21]:
# Instalar si es necesario: !pip install mlxtend
import pandas as pd
import re
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. INGESTA DE DATOS
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
df_spotify = pd.read_csv(url, encoding='latin-1')

# 2. LIMPIEZA: Eliminar nulos de la columna de artistas
df_spotify = df_spotify.dropna(subset=['Artist']).copy()
print(f"Canciones disponibles: {len(df_spotify)}")
print("Columnas del dataset:", df_spotify.columns.tolist())
print("Vista de las primeras 10 canciones y sus datos:")
display(df_spotify.head(10))

# 3. TRANSFORMACIÓN: Separar los artistas que colaboran en cada canción
# Asumimos que las colaboraciones están separadas por coma (ej. "Bad Bunny, Feid")
# Convertimos esa columna en una verdadera lista de Python

# se tuvo que definir una funcion para separar a los artistas ya que unos tienen simbolos como & o feat. o featuring
def separar_artistas(cadena):
    artistas = re.split(r',|&|\bfeat\b|\bfeat\.\b|\bfeaturing\b', cadena, flags=re.IGNORECASE)
    return [a.strip() for a in artistas if a.strip()]

# se modifico con "separar_artistas).tolist()" para realizar la separacion correcta
lista_colaboraciones = df_spotify['Artist'].astype(str).apply(separar_artistas).tolist()
    
#separar_artistas).tolist()
#lambda x: [artista.strip() for artista in x.split(',') if artista.strip()]).tolist()

# Muestra rápida de cómo se ven nuestras "canastas"
print("Muestra de las primeras 5 colaboraciones:")
for i, artistas in enumerate(lista_colaboraciones[:5], start=1):
    print(f"Canción {i}: {artistas}")


Canciones disponibles: 4595
Columnas del dataset: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']
Vista de las primeras 10 canciones y sus datos:


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,TikTok Posts,TikTok Likes,TikTok Views,YouTube Playlist Reach,Apple Music Playlist Count,AirPlay Spins,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",92.0,"84,274,754","1,713,126","5,767,700","651,565,900","5,332,281,936","150,597,040",210.0,"40,975",684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",92.0,"116,347,040","3,486,739","674,700","35,223,547","208,339,025","156,380,351",188.0,"40,778",3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",92.0,"122,599,116","2,228,730","3,025,400","275,154,237","3,369,120,610","373,784,955",190.0,"74,333",536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0
3,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,USSM12209777,4,444.9,"2,031,280,633","269,802","136,569,078",85.0,"1,096,100,899","10,629,796","7,189,811","1,078,757,968","14,603,725,994","3,351,188,582",394.0,"1,474,799","2,182",264.0,"24,684,248",210.0,"190,260,277","203,384",NaN,"11,822,942",NaN,0
4,Houdini,Houdini,Eminem,5/31/2024,USUG12403398,5,423.3,"107,034,922","7,223","151,469,874",88.0,"77,373,957","3,670,188","16,400",NaN,NaN,"112,763,851",182.0,"12,185",1,82.0,"17,660,624",105.0,"4,493,884","7,006","207,179","457,017",NaN,1
5,Lovin On Me,Lovin On Me,Jack Harlow,11/10/2023,USAT22311371,6,410.1,"670,665,438","105,892","175,421,034",83.0,"131,148,091","1,392,593","4,202,367","214,943,489","2,938,686,633","2,867,222,632",138.0,"522,042","4,654",86.0,"17,167,254",152.0,"138,529,362","50,982","9,438,601","4,517,131",NaN,1
6,Beautiful Things,Beautiful Things,Benson Boone,1/18/2024,USWB12307016,7,407.2,"900,158,751","73,118","201,585,714",86.0,"308,723,145","4,120,760",NaN,"29,584,940","534,915,313","4,601,579,812",280.0,"383,478",429,168.0,"48,197,850",154.0,"65,447,476","57,372",NaN,"9,990,302",NaN,0
7,Gata Only,Gata Only,FloyyMenor,2/2/2024,QZL382406049,8,375.8,"675,079,153","40,094","211,236,940",92.0,"228,382,568","1,439,495","3,500,000","338,546,668","3,804,584,163","2,112,581,620",160.0,"17,221",30,87.0,"33,245,595",53.0,"3,372,428","5,762",NaN,"6,063,523",NaN,1
8,Danza Kuduro - Cover,ýýýýýýýýýýýýýýýýýýýýý - ýýýýýýýýýýýýýýýýýý -,MUSIC LAB JPN,6/9/2024,TCJPA2463708,9,355.7,"1,653,018,119",1,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
9,BAND4BAND (feat. Lil Baby),BAND4BAND (feat. Lil Baby),Central Cee,5/23/2024,USSM12404354,10,330.6,"90,676,573","10,400","184,199,419",86.0,"32,735,244","988,682","325,800","121,574,500","974,656,200","174,706,874",191.0,"3,823",117,78.0,"10,800,098",92.0,"1,005,626",842,"3,679,709","666,302",NaN,1


Muestra de las primeras 5 colaboraciones:
Canción 1: ['Tommy Richman']
Canción 2: ['Kendrick Lamar']
Canción 3: ['Artemas']
Canción 4: ['Miley Cyrus']
Canción 5: ['Eminem']


In [22]:
# Utiliza TransactionEncoder para convertir 'lista_colaboraciones' en una matriz booleana (One-Hot)
# Guárdala en un DataFrame llamado 'df_transacciones'

te = TransactionEncoder()
matriz_bool = te.fit(lista_colaboraciones).transform(lista_colaboraciones)
df_transacciones = pd.DataFrame(matriz_bool, columns=te.columns_)

### Fase 2: Aplicando Apriori
Ahora que tenemos la matriz con ceros y unos, extraigamos los conjuntos de artistas que colaboran frecuentemente.
Debido a que hay miles de artistas solistas, las colaboraciones son estadísticamente raras. Usaremos un `min_support` muy bajo para capturar las colaboraciones.

In [33]:
# --- RETO PARA EL ALUMNO AQUÍ ---

# el trabajo indica que se necesita realizar un soporte minimo de 0.005, al realizar esto no muestra
# ningun resultado, por lo que se decide bajar el soporte minimo a 0.0004 para obtener resultados

# no estoy seguro si hay una solucion para forzar los resultados teniendo como soporte minimo 0.005
# aunque se utilizo el mismo dataset para hacer la tabla pero con un soporte MAS bajo para ver los datos

# 1. Encuentra los Itemsets Frecuentes usando un soporte mínimo más bajo (0.0004 o 0.0005)
itemsets_frecuentes = apriori(df_transacciones, min_support=0.0004, use_colnames=True)

# 2. Genera las Reglas de Asociación filtrando por la métrica 'lift' con un mínimo de 1.2
reglas_musicales = association_rules(itemsets_frecuentes, metric="lift", min_threshold=1.2)

# 3. Muestra las reglas resultantes ordenadas por 'confidence' de mayor a menor
display(reglas_musicales.sort_values('confidence', ascending=False))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Ayo),(Teo),0.000435,0.000435,0.000435,1.0,2297.500000,1.0,0.000435,inf,1.000000,1.000000,1.000000,1.00
1,(Teo),(Ayo),0.000435,0.000435,0.000435,1.0,2297.500000,1.0,0.000435,inf,1.000000,1.000000,1.000000,1.00
2,(Status),(Chase),0.000435,0.000435,0.000435,1.0,2297.500000,1.0,0.000435,inf,1.000000,1.000000,1.000000,1.00
3,(Chase),(Status),0.000435,0.000435,0.000435,1.0,2297.500000,1.0,0.000435,inf,1.000000,1.000000,1.000000,1.00
4,(Lennox),(Zion),0.000653,0.000653,0.000653,1.0,1531.666667,1.0,0.000652,inf,1.000000,1.000000,1.000000,1.00
5,(Zion),(Lennox),0.000653,0.000653,0.000653,1.0,1531.666667,1.0,0.000652,inf,1.000000,1.000000,1.000000,1.00
6,(Ryan Lewis),(Macklemore),0.000435,0.000871,0.000435,1.0,1148.750000,1.0,0.000435,inf,0.999565,0.500000,1.000000,0.75
8,(Tyler),(The Creator),0.001523,0.001523,0.001523,1.0,656.428571,1.0,0.001521,inf,1.000000,1.000000,1.000000,1.00
9,(The Creator),(Tyler),0.001523,0.001523,0.001523,1.0,656.428571,1.0,0.001521,inf,1.000000,1.000000,1.000000,1.00
7,(Macklemore),(Ryan Lewis),0.000871,0.000435,0.000435,0.5,1148.750000,1.0,0.000435,1.999129,1.000000,0.500000,0.499782,0.75


### 🧠 Fase 3: Interpretación de Negocio (El Entregable)

Como Científico de Datos, no basta con entregar una tabla. Debes explicarle los hallazgos al Director Creativo. Revisa tu DataFrame `reglas_musicales` y responde las siguientes 4 preguntas:

**1. Análisis de Soporte (El Volumen):**
Menciona a los dos artistas que conforman la regla con el mayor **Soporte**. ¿Qué significa este número en términos simples para el negocio?

> **Tu Respuesta:** [Escribe aquí]

**2. Análisis de Confianza (La Dirección):**
Busca una regla que tenga una **Confianza** mayor al 0.5 (50%). Traduce esta regla matemática a una frase que el director pueda entender (Ej. "Si contratamos al artista A, existe un X% de probabilidad de que el artista B también deba estar en el track para que sea el éxito que registramos").

> **Tu Respuesta:** [Escribe aquí]

**3. Análisis de Lift (El Impulso):**
Busca la regla con el **Lift** más alto de toda tu tabla. 
¿Por qué esta métrica es más confiable que el Soporte para decidir si estos dos artistas *realmente* tienen química musical o si solo fue una coincidencia estadística?

> **Tu Respuesta:** [Escribe aquí]

**4. Toma de Decisión:**
Basado puramente en las reglas matemáticas que descubriste, si la disquera solo tiene presupuesto para juntar a dos artistas en el estudio la próxima semana, ¿qué pareja elegirías y por qué? (Justifica usando las 3 métricas de oro).

> **Tu Respuesta:** [Escribe aquí]